# `Leakly` usage example
---

This example demonstrates how to use the `Leakly` package to perform a permutation test for checking potential leakage.

In this example,
 - We will synthesize data;
 - Run permutation tests with our exemplar machine learning pipeline;
 - Create a summary plot to interpret the results.


## Table of Contents
- [1. Synthesizing Data](#1-synthesizing-data)
- [2. Defining the Machine Learning Pipeline](#2-defining-the-machine-learning-pipeline)
- [3. Running Permutation Tests](#3-running-permutation-tests)
- [4. Visualizing Results](#4-visualizing-results)

In [ ]:
# Install Leakly and its notebook dependencies in a fresh Codespace.
%pip install -e ".[notebook]"


## 1. Synthesizing Data
We will start by synthesizing a dataset that simulates a classification problem. We will use the `simulate_dataset` function from the `Leakly` package to create a dataset with a specified configuration.

In [ ]:
# Import libraries.
from leakly import SimulationConfig, simulate_dataset

simulated_data = simulate_dataset(
    SimulationConfig(
        n_samples=200, # number of samples in the dataset
        n_features=1000, # number of features in the dataset
        n_covariates=3, # number of covariates in the dataset
        effect_fraction=0.1, # fraction of features that have an effect
        effect_size=0.5, # size of the effect for the informative features
        class_balance=0.5, # proportion of positive class
        random_state=42, # random seed for reproducibility
    )
)
# User could replace with their X, y, and covariates here.
X, y, covariates = \
    simulated_data.X, simulated_data.y, simulated_data.covariates


## 2. Defining the Machine Learning Pipeline

Next, we will define a machine learning pipeline. 

Users can also bring their own pipeline, as long as it returns a test score from `X`, `y`, and optional covariates. 

Here, we use example random-forest pipelines to classify the data.

In [ ]:
from leakly import load_example_leakage_config, load_example_nonleakage_config
from leakly import print_config

leakage_config = load_example_leakage_config()
nonleakage_config = load_example_nonleakage_config()

# Users can inspect and modify the config.
print_config(leakage_config)
# Modify the config as needed.
leakage_config["imputation"]["n_neighbors"] = 4
print_config(leakage_config)

In [ ]:
# Users can put their own pipeline here,
# as long as it generates a test score from X, y, and optional covariates.
from leakly import MLPipeline

leakage_pipeline = MLPipeline(X, y, covariates=covariates, config=leakage_config)
nonleakage_pipeline = MLPipeline(X, y, covariates=covariates, config=nonleakage_config)

# check test score (AUC in this case) for both pipelines
leakage_score = (leakage_pipeline.fit()).evaluate()
nonleakage_score = (nonleakage_pipeline.fit()).evaluate()

print(f"Leakage Pipeline Score: {leakage_score:.3f}")
print(f"Non-Leakage Pipeline Score: {nonleakage_score:.3f}")

## 3. Running Permutation Tests

Once we have our data and machine learning pipeline defined, we can run permutation tests to evaluate the performance of our model and check for potential data leakage.

Since we have two pipelines (leakage and non-leakage), we will run permutation tests for both pipelines to compare their performance.

In [ ]:
from leakly import permute_label
from tqdm.auto import tqdm

# Use 2000 permutations for now to replicate the `AUC.png`.
# Adjust based on your computational resources and needs.
N_PERMUTATIONS = 2,000

leakage_permuted_scores = []
nonleakage_permuted_scores = []

for i in tqdm(range(N_PERMUTATIONS), desc="Running permutations"):
    permuted_y = permute_label(y, random_state=i)  # Permute labels
    # define pipelines with permuted labels
    # user could change with their own pipeline
    leakage_pipeline_perm = MLPipeline(X, permuted_y,
                                       covariates=covariates,
                                       config=leakage_config)
    nonleakage_pipeline_perm = MLPipeline(X, permuted_y,
                                          covariates=covariates,
                                          config=nonleakage_config)

    # fit and get test scores for both pipelines with permuted labels
    leakage_score_perm = (leakage_pipeline_perm.fit()).evaluate()
    nonleakage_score_perm = (nonleakage_pipeline_perm.fit()).evaluate()

    leakage_permuted_scores.append(leakage_score_perm)
    nonleakage_permuted_scores.append(nonleakage_score_perm)


## 4. Visualizing Results

Finally, we will use the `SummaryPlotter` class from the `Leakly` package to create a summary plot of the permutation test results. This plot will help us visualize the distribution of test scores under the null hypothesis and compare it to the observed test score.

In [ ]:
from leakly import SummaryPlotter

leakage_plotter = SummaryPlotter(leakage_permuted_scores, chance_level=0.5)
leakage_plotter.plot()

nonlocal_plotter = SummaryPlotter(nonleakage_permuted_scores, chance_level=0.5)
nonlocal_plotter.plot()

The leakage pipeline can keep a high AUC even after labels are permuted.
The non-leakage pipeline should stay close to chance level after permutation.